# 인공지능 Term Project 13조 — 경상도 사투리 → 표준어 → 중국어 번역
**사투리 문장 → 표준어 문장 변환(KoBART 미세조정) → 외국어 번역(MADLAD, 사전학습 그대로)** — AI Hub 경상도 방언 데이터 (학습쌍 약 56만 개)

실습 전 준비 (Anaconda Prompt):
```
conda activate (본인 가상환경)
pip install torch --index-url https://download.pytorch.org/whl/cu128
pip install -r requirements.txt   # matplotlib, gradio 포함
python scripts/build_pairs.py          # data/pairs/*.csv 만들기 (AI Hub 데이터가 있을 때)
```
위 명령 대신 바로 아래 **설치 셀**을 실행해도 됩니다 (없는 패키지만 설치).

AI Hub 데이터가 없는 PC(교수님 확인용)에서는 함께 제출한 **`data_sample/`(학습쌍 약 6천 개)** 폴더를 저장소 최상위에 두면 자동으로 그걸 씁니다.
학습된 모델로 바로 결과만 보려면 공유한 모델 폴더를 `checkpoints/kobart_norm/best/`에 두고 0장 설정에서 `TRAIN = False`로 바꾸세요.

이 노트북을 **저장소 최상위 폴더**(README.md가 있는 곳)에 두고, 각 셀을 위에서부터 순서대로 `Shift + Enter`로 실행하세요. GPU(RTX 3060 Ti 8GB 기준)가 필요합니다.

| 장 | 내용 | 강의 개념 |
|---|---|---|
| 0 | 준비: 라이브러리, GPU 확인, 실험 설정 | |
| 1~2 | 데이터 살펴보기 → 입력 X(사투리) / 정답 y(표준어) → **화자 기준** 훈련·검증·시험 분할 | 지도학습의 데이터셋, 훈련/테스트 분할 |
| 3 | 기준선: "아무것도 안 바꾸고 그대로 복사"하면 몇 점인가 | 평가 지표 (chrF·CER) |
| 4~5 | **KoBART 미세조정**: 토큰화 → 손실 함수 → 경사하강법(AdamW) → 손실 곡선 | 손실 함수·경사하강법·학습률 |
| 6 | 예측과 평가: 시험 세트에서 모델 vs 복사 기준선 | 테스트 세트 평가 |
| 7 | 번역 비교 실험 A/B/C: 사투리 그대로 / 모델 표준어 / 정답 표준어 → 중국어 | 비교 실험 설계 |
| 8 | Gradio 데모 화면 | |
| 9 | (도전) 학습률 바꿔 보기 | 학습률 |

> **실행 시간**: 5장 학습이 가장 오래 걸립니다 (5만 쌍·3에폭 기준 1~2시간). 이미 학습한 모델이 있으면 `TRAIN = False`로 두면 불러오기만 합니다.
>
> **주의**: 출력 칸에 AI Hub 문장이 찍힙니다. AI Hub 데이터는 재배포 금지이므로, 이 노트북을 GitHub에 올릴 때는 **메뉴 → 모든 출력 지우기** 후 커밋하세요.

## 설치 — 처음 한 번만
노트북에 필요한 패키지가 있는지 확인하고, **없는 것만** 설치합니다. 이미 다 있으면 아무것도 하지 않습니다.
- NVIDIA GPU가 있으면 torch는 GPU(CUDA) 버전으로 설치합니다.
- 설치만 확인하고 싶으면 `AUTO_INSTALL = False`로 바꾸고 실행하세요 (빠진 것 목록만 출력).
- 무언가 설치했다면 **메뉴 → 커널 다시 시작** 후 아래 0장부터 실행하세요.

In [ ]:
AUTO_INSTALL = True            # False → 빠진 패키지 목록만 보여 주고 설치하지 않음

import importlib, importlib.util, shutil, subprocess, sys

TORCH_CUDA_INDEX = "https://download.pytorch.org/whl/cu128"   # CUDA 12.x용 torch
NEED = {                        # 불러올 이름: pip 설치 이름
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "transformers": "transformers>=4.44", "sentencepiece": "sentencepiece", "google.protobuf": "protobuf",
    "sacrebleu": "sacrebleu", "jiwer": "jiwer", "gradio": "gradio",
}

def has(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except ModuleNotFoundError:
        return False

def pip(*args):
    print("$ pip install", " ".join(args))
    subprocess.run([sys.executable, "-m", "pip", "install", *args], check=True)

gpu = shutil.which("nvidia-smi") is not None
missing = [pkg for mod, pkg in NEED.items() if not has(mod)]
need_torch = not has("torch")
print("NVIDIA GPU:", "있음" if gpu else "없음 (CPU로는 학습이 매우 느림)")
print("torch:", "설치 필요" if need_torch else "있음")
print("그 밖에 빠진 패키지:", missing or "없음")

if (need_torch or missing) and not AUTO_INSTALL:
    print("\nAUTO_INSTALL = False → 설치하지 않음. 위 목록을 직접 설치하세요.")
elif need_torch or missing:
    if need_torch:                                        # torch를 먼저 (GPU 버전이 깔리도록)
        pip("torch", *(["--index-url", TORCH_CUDA_INDEX] if gpu else []))
    if missing:
        pip(*missing)
    importlib.invalidate_caches()
    print("\n설치 끝 → 메뉴에서 '커널 다시 시작' 후 0장부터 실행하세요.")
else:
    print("\n필요한 패키지가 모두 있음 → 바로 0장부터 실행하면 됩니다.")

if not need_torch:                                        # 이미 있는 torch가 CPU 버전인지 확인
    import torch
    if gpu and not torch.cuda.is_available():
        print(f"\n[!] torch {torch.__version__}가 CPU 버전입니다. GPU를 쓰려면 Anaconda Prompt에서:\n"
              f"    pip install --force-reinstall torch --index-url {TORCH_CUDA_INDEX}")

## 0. 준비 — 라이브러리 불러오기와 실험 설정
`torch`(딥러닝) · `transformers`(사전학습 모델) · `sacrebleu`·`jiwer`(번역·변환 평가 지표) · `pandas`·`matplotlib`(표·그래프)

In [ ]:
import math, random, time, difflib
from pathlib import Path

import numpy as np                                        # 수치 계산
import pandas as pd                                       # CSV 읽기, 데이터프레임(표)
import matplotlib.pyplot as plt                           # 그래프
import matplotlib.font_manager as fm

import torch                                              # 딥러닝 프레임워크 (GPU 계산)
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
import sacrebleu                                          # BLEU, chrF (번역 평가 표준 도구)
import jiwer                                              # CER (글자 오류율)

# 그래프 한글 글꼴 (Windows: 맑은 고딕)
for f in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
    if any(f == x.name for x in fm.fontManager.ttflist):
        plt.rcParams["font.family"] = f
        break
plt.rcParams["axes.unicode_minus"] = False                # 마이너스 기호 깨짐 방지

random.seed(42); np.random.seed(42); torch.manual_seed(42)  # 매번 같은 결과가 나오도록 난수 고정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__)
if device == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, VRAM {p.total_memory / 2**30:.1f}GB")
else:
    print("[!] GPU가 없음 → 학습이 매우 느립니다. torch가 CPU 버전으로 깔렸는지 확인")

In [ ]:
# ---------------- 실험 설정 (여기만 바꾸면 됨) ----------------
# 학습쌍 CSV 폴더: 전체 데이터(data/pairs)가 있으면 그걸, 없으면 공유용 샘플(data_sample/pairs, 약 6천 쌍)을 쓴다
PAIRS_DIR  = Path("data/pairs") if Path("data/pairs/train.csv").exists() else Path("data_sample/pairs")
MODEL_NAME = "gogamza/kobart-base-v2"                     # 사전학습 KoBART (한국어 BART, 약 1.2억 파라미터)
MT_MODEL   = "google/madlad400-3b-mt"                     # 번역 모델 (학습 안 함, 그대로 사용)
CKPT_DIR   = Path("checkpoints/kobart_norm_nb")           # 이 노트북에서 학습한 모델 저장 위치

TRAIN    = True                                           # False → 학습 건너뛰고 저장된 모델 불러오기
N_TRAIN  = 50000                                          # 학습에 쓸 쌍 수 (전체 46만 중 일부로 시작)
EPOCHS   = 3                                              # 훈련 세트를 몇 번 반복해서 볼지
BATCH    = 16                                             # 한 번에 보는 문장 수 (GPU 메모리 부족하면 8)
LR       = 5e-5                                           # 학습률 (사전학습 모델 미세조정은 작게)
MAX_LEN  = 128                                            # 문장 최대 토큰 수
N_VALID  = 500                                            # 에폭마다 생성 품질을 볼 검증 문장 수
N_TEST   = 1000                                           # 6장에서 평가할 시험 문장 수
DECODE   = {"num_beams": 4, "max_new_tokens": 128, "do_sample": False}   # 생성(디코딩) 설정 — 고정
MT_DECODE = {"num_beams": 4, "max_new_tokens": 256, "do_sample": False}  # 번역 디코딩 설정 — evaluate.py와 같게 고정

print("데이터:", PAIRS_DIR, "(전체)" if PAIRS_DIR.parts[0] == "data" else "(공유용 샘플 → 학습은 짧게 끝나지만 점수는 전체 데이터보다 낮게 나옴)")

## 1. 데이터 불러오기와 살펴보기
`scripts/build_pairs.py`가 AI Hub 라벨(JSON)에서 **사투리 문장과 표준어 문장 쌍**만 뽑아 정제·분할해 둔 결과를 불러옵니다.

In [ ]:
train_df = pd.read_csv(PAIRS_DIR / "train.csv").fillna("")   # 훈련 세트
valid_df = pd.read_csv(PAIRS_DIR / "valid.csv").fillna("")   # 검증 세트 (학습 중 점검용)
test_df  = pd.read_csv(PAIRS_DIR / "test.csv").fillna("")    # 시험 세트 (마지막 평가에만 사용)
print("크기(행, 열):", train_df.shape, valid_df.shape, test_df.shape)
train_df[["dataset", "subset", "n_eojeol", "dialect", "standard"]].head()  # 앞 5행: dialect(사투리) → standard(표준어)

In [ ]:
# 데이터 종류별·분할별 개수 표
SUBSET_NAME = {"conv": "014 일상대화", "st": "노년층 따라말하기", "say": "노년층 질문답하기", "talk": "노년층 2인대화"}
all_df = pd.concat([train_df, valid_df, test_df], ignore_index=True)
all_df["종류"] = all_df.subset.map(SUBSET_NAME)
table = all_df.pivot_table(index="종류", columns="split", values="id", aggfunc="count")[["train", "valid", "test"]]
table["합계"] = table.sum(axis=1)
table.loc["합계"] = table.sum()
table

In [ ]:
# 정제 단계별로 남은 발화 수 → 원본 274만 개 중 '사투리와 표준어가 다른' 쌍만 남는다
steps = pd.read_csv(PAIRS_DIR / "filter_steps.csv", index_col=0)
plt.figure(figsize=(8, 4))
plt.barh(steps.index[::-1], steps["전체"][::-1] / 10000, color="steelblue")   # 위에서 아래로 단계 순서
plt.xlabel("남은 발화 수 (만 개)")
plt.title("정제 단계별 남은 발화 수")
plt.grid(True, axis="x")
plt.tight_layout()
plt.show()
# 가장 크게 줄어든 곳: '방언 = 표준어' 쌍 제거 → 그대로 두면 모델이 입력을 베끼는 것만 배운다

In [ ]:
# 문장 길이(어절 수) 분포 — MAX_LEN을 정할 때 참고
plt.figure(figsize=(6, 4))
plt.hist(train_df.n_eojeol.clip(upper=60), bins=60, color="red", alpha=0.7)
plt.xlabel("사투리 문장 어절 수 (60 이상은 60으로)")
plt.ylabel("문장 수")
plt.title(f"어절 수 분포 (중앙값 {train_df.n_eojeol.median():.0f})")
plt.grid(True)
plt.show()

In [ ]:
def changes(d, s):
    # 어절 단위로 비교해서 달라진 부분만 '사투리 → 표준어'로 보여준다
    a, b = str(d).split(), str(s).split()
    out = []
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b).get_opcodes():
        if op != "equal":
            out.append(f"{' '.join(a[i1:i2]) or '∅'} → {' '.join(b[j1:j2]) or '∅'}")
    return " / ".join(out)

# 종류별로 2개씩 예시
ex = train_df.groupby("subset").sample(2, random_state=0)
for r in ex.itertuples():
    print(f"[{SUBSET_NAME[r.subset]}] {r.dialect}\n    바뀐 부분: {changes(r.dialect, r.standard)}")

## 2. 입력 X 와 정답 y — 그리고 **화자 기준** 분할
지도학습 데이터셋 $D=\{(x_i, y_i)\}$: 입력 $x$(사투리 문장) → 정답 $y$(표준어 문장). 강의의 `train_test_split`은 **문장을 무작위로** 섞지만,
여기서는 **같은 화자(대화 파일)의 문장이 훈련과 시험에 동시에 들어가지 않도록** 화자 단위로 나눴습니다. 같은 사람 말투를 훈련에서 보고 시험을 보면 점수가 부풀려지기 때문입니다.

In [ ]:
X_train, y_train = train_df.dialect.tolist(), train_df.standard.tolist()   # 입력(사투리), 정답(표준어)
X_test,  y_test  = test_df.dialect.tolist(),  test_df.standard.tolist()

# 확인 1: 훈련과 시험에 같이 나오는 화자 그룹이 있나? → 0이어야 함
print("훈련∩시험 화자 그룹:", len(set(train_df.group) & set(test_df.group)), "개")
# 확인 2: 시험 문장이 훈련 데이터에 그대로 있나? → 0이어야 함 (build_pairs.py 9단계에서 제거)
print("훈련에 똑같이 있는 시험 사투리 문장:", len(set(test_df.dialect) & set(train_df.dialect)), "개")
print(f"훈련 {len(X_train):,}쌍   검증 {len(valid_df):,}쌍   시험 {len(X_test):,}쌍")

## 3. 기준선 — "아무것도 안 바꾸고 그대로 복사"하면 몇 점인가
사투리와 표준어는 대부분의 어절이 같습니다. 그래서 **입력을 그대로 내놓기만 해도 점수가 꽤 높게** 나옵니다 (Park 외, LREC 2026이 지적한 BLEU의 함정).
우리 모델은 이 **복사 기준선보다 높아야** 실제로 뭔가 고친 것입니다.

| 지표 | 뜻 | 좋은 방향 |
|---|---|---|
| chrF | 글자 n-gram이 정답과 얼마나 겹치나 (0~100) | 높을수록 |
| BLEU(글자) | 글자 단위 BLEU (0~100) | 높을수록 |
| CER | 정답 대비 글자 오류율 = (바꾸기+지우기+넣기) / 정답 글자 수 | 낮을수록 |
| 완전일치 | 정답과 글자 하나까지 같은 문장 비율 | 높을수록 |

In [ ]:
import re
def norm_key(t):
    return re.sub(r"[^\w]", "", str(t))                   # 비교용: 공백·문장부호 제거

def scores(pred, ref):
    # 변환 품질 지표 4개를 한 번에 계산
    p, r = [norm_key(x) for x in pred], [norm_key(x) for x in ref]
    return {
        "chrF": round(sacrebleu.corpus_chrf(list(pred), [list(ref)]).score, 2),
        "BLEU_char": round(sacrebleu.corpus_bleu(list(pred), [list(ref)], tokenize="char").score, 2),
        "CER": round(jiwer.cer(r, p), 4),
        "완전일치": round(sum(a == b for a, b in zip(p, r)) / len(r), 4),
    }

test_s = test_df.sample(n=min(N_TEST, len(test_df)), random_state=42).reset_index(drop=True)   # 평가용 시험 문장
copy_score = scores(test_s.dialect, test_s.standard)     # 예측 = 입력 그대로
print("복사 기준선 (시험", len(test_s), "문장):", copy_score)

## 4. 모델 만들기 — 사전학습 KoBART 불러오기와 토큰화
KoBART는 **encoder-decoder** 구조입니다: 인코더가 사투리 문장을 읽고, 디코더가 표준어 문장을 한 토큰씩 생성합니다.
처음부터 학습하지 않고, 한국어를 이미 배운 모델을 우리 데이터로 **미세조정(fine-tuning)** 합니다.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)                       # 토크나이저: 문장 → 토큰 ID
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)  # 사전학습 가중치를 GPU로
if model.config.decoder_start_token_id is None:
    model.config.decoder_start_token_id = tok.eos_token_id
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

s = "쪼매만 기다리래이"
print("토큰:", tok.tokenize(s))                                        # 문장이 어떻게 쪼개지나
print("ID  :", tok(s)["input_ids"])                                    # KoBART는 문장 끝 </s>를 자동으로 안 붙인다 → 아래에서 직접 붙임

In [ ]:
def encode(texts, max_len=MAX_LEN):
    # 문장 목록 → 토큰 ID 목록. 끝에 </s>(문장 끝 표시)를 붙이고 너무 길면 자른다
    eos, out = tok.eos_token_id, []
    for t in texts:
        ids = tok(str(t), add_special_tokens=True, return_token_type_ids=False)["input_ids"]
        if ids and ids[-1] == eos:
            ids = ids[:-1]
        out.append(ids[: max_len - 1] + [eos])
    return out

def pad(seqs, label=False):
    # 길이가 다른 문장들을 같은 길이로 맞춘다. 정답(label)의 빈자리는 -100 → 손실 계산에서 제외
    n = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), n), -100 if label else tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), n), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    return ids, mask

@torch.inference_mode()
def normalize(texts, bs=16):
    # 사투리 문장 목록 → 모델이 생성한 표준어 문장 목록 (빔 서치 4)
    model.eval()
    outs = []
    for i in range(0, len(texts), bs):
        x, m = pad(encode(texts[i:i + bs]))
        gen = model.generate(input_ids=x.to(device), attention_mask=m.to(device),
                             pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id, **DECODE)
        outs += [o.strip() for o in tok.batch_decode(gen, skip_special_tokens=True)]
        if device == "cuda":
            torch.cuda.empty_cache()   # 길이가 다른 묶음마다 메모리가 쌓이면 8GB GPU가 넘쳐서 수십 배 느려진다
    return outs

print("학습 전 KoBART 출력:", normalize(["니 밥 뭇나"]))           # 아직 사투리 변환을 배우지 않은 상태

## 5. 학습 — 손실 함수와 경사하강법 (KoBART 미세조정)
강의의 선형 회귀와 **같은 틀**입니다. 다만 파라미터가 2개($w_0, w_1$)가 아니라 약 1.2억 개입니다.

- 모델: $\hat{y} = f_\theta(x)$ — 사투리 $x$를 보고 표준어 토큰을 하나씩 예측
- 손실 함수: **교차 엔트로피** $L(\theta) = -\dfrac{1}{T}\sum_t \log P_\theta(y_t \mid y_{<t}, x)$ — 정답 토큰에 준 확률이 낮을수록 손실이 큼
- 업데이트: $\theta \leftarrow \theta - \alpha \nabla_\theta L$ — 기울기는 `loss.backward()`가 자동으로 계산 (역전파)
- 옵티마이저: **AdamW** (경사하강법의 개선판), 학습률은 처음 5% 동안 올렸다가(warm-up) 직선으로 줄임

| 강의 (선형 회귀) | 여기 (KoBART) |
|---|---|
| `gradients()` 직접 계산 | `loss.backward()` 자동 미분 |
| `w = w - lr * g` | `optimizer.step()` |
| 전체 데이터 한 번에 | 미니배치(16문장)씩 |

In [ ]:
def batches(src, tgt, bs, shuffle=True):
    # (사투리, 표준어) 쌍을 미니배치로 묶어서 하나씩 돌려준다
    idx = list(range(len(src)))
    if shuffle:
        random.shuffle(idx)
    for i in range(0, len(idx), bs):
        b = idx[i:i + bs]
        x, m = pad(encode([src[j] for j in b]))
        y, _ = pad(encode([tgt[j] for j in b]), label=True)
        yield x.to(device), m.to(device), y.to(device)

amp = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else None   # bf16: 메모리 절약·속도 향상
history, val_hist = [], []                                # 손실 기록, 에폭별 검증 점수 기록
valid_s = valid_df.sample(n=min(N_VALID, len(valid_df)), random_state=1).reset_index(drop=True)
valid_copy = scores(valid_s.dialect, valid_s.standard)["chrF"]

if TRAIN:
    tr = train_df.sample(n=min(N_TRAIN, len(train_df)), random_state=42)
    src, tgt = tr.dialect.tolist(), tr.standard.tolist()
    steps_per_epoch = math.ceil(len(src) / BATCH)
    total = steps_per_epoch * EPOCHS
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)          # 경사하강법(AdamW)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total * 0.05), total)      # 학습률 스케줄
    print(f"학습 {len(src):,}쌍 × {EPOCHS}에폭 = {total:,} 스텝")
    best, t0, step = -1, time.time(), 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for x, m, y in batches(src, tgt, BATCH):
            with torch.autocast("cuda", dtype=amp, enabled=amp is not None):
                loss = model(input_ids=x, attention_mask=m, labels=y).loss   # ① 순전파: 손실 계산
            loss.backward()                                                  # ② 역전파: 기울기 계산
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)          #    기울기가 너무 크면 잘라서 안정화
            optimizer.step(); scheduler.step(); optimizer.zero_grad()        # ③ 파라미터 업데이트 → 기울기 초기화
            history.append(loss.item()); step += 1
            if step % 200 == 0:
                el = time.time() - t0
                print(f"  epoch {epoch} step {step:5d}/{total}  loss {np.mean(history[-200:]):.4f}"
                      f"  남은 시간 약 {el / step * (total - step) / 60:.0f}분")
        pred = normalize(valid_s.dialect.tolist())                           # 에폭이 끝날 때마다 검증
        sc = scores(pred, valid_s.standard)
        val_hist.append(sc["chrF"])
        mark = ""
        if sc["chrF"] > best:                                                # 검증 chrF가 가장 높은 모델만 저장
            best, mark = sc["chrF"], "  ← best 저장"
            model.save_pretrained(CKPT_DIR); tok.save_pretrained(CKPT_DIR)
        print(f"[epoch {epoch}] 검증 chrF {sc['chrF']} (복사 {valid_copy})  CER {sc['CER']}  완전일치 {sc['완전일치']}{mark}")
    model = AutoModelForSeq2SeqLM.from_pretrained(CKPT_DIR).to(device)       # best 모델로 되돌리기
    del optimizer, scheduler                                                 # 스케줄러도 옵티마이저를 잡고 있어서 같이 지운다
    import gc; gc.collect(); torch.cuda.empty_cache()
else:
    load = CKPT_DIR if (CKPT_DIR / "config.json").exists() else Path("checkpoints/kobart_norm/best")
    model = AutoModelForSeq2SeqLM.from_pretrained(load).to(device)
    print("저장된 모델 불러옴:", load)

In [ ]:
# 손실 곡선과 에폭별 검증 점수 — 손실이 내려가고, 검증 chrF가 복사 기준선 위로 올라가야 정상
if history:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sm = pd.Series(history).rolling(100, min_periods=1).mean()        # 100스텝 이동평균 (들쭉날쭉한 걸 매끈하게)
    axes[0].plot(history, alpha=0.25, label="스텝별")
    axes[0].plot(sm, color="red", label="이동평균")
    axes[0].set_xlabel("step"); axes[0].set_ylabel("cross-entropy loss"); axes[0].set_title("손실 곡선"); axes[0].legend()
    axes[1].plot(range(1, len(val_hist) + 1), val_hist, "o-", label="모델")
    axes[1].axhline(valid_copy, color="gray", ls="--", label="복사 기준선")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("검증 chrF"); axes[1].set_title("에폭별 검증 점수"); axes[1].legend()
    for ax in axes:
        ax.grid(True)
    plt.tight_layout(); plt.show()

## 6. 예측과 평가 — 시험 세트에서 모델 vs 복사 기준선
**학습에 쓰지 않은 화자**의 시험 문장으로 평가합니다.

In [ ]:
t0 = time.time()
test_s["pred"] = normalize(test_s.dialect.tolist())                 # 시험 문장 → 모델 표준어
print(f"{len(test_s)}문장 변환 {time.time() - t0:.0f}초")
model_score = scores(test_s.pred, test_s.standard)
pd.DataFrame([copy_score, model_score], index=["복사 기준선 (입력 그대로)", "KoBART 미세조정"])

In [ ]:
# 종류별 chrF — 어떤 사투리에서 개선이 크고 어디서 약한가
rows = []
for sub, g in test_s.groupby("subset"):
    rows.append({"종류": SUBSET_NAME[sub], "문장 수": len(g),
                 "복사": scores(g.dialect, g.standard)["chrF"], "모델": scores(g.pred, g.standard)["chrF"]})
by = pd.DataFrame(rows).set_index("종류")
by[["복사", "모델"]].plot.bar(figsize=(7, 4), rot=0, color=["gray", "red"])
plt.ylabel("chrF"); plt.title("종류별 변환 점수 (시험 세트)"); plt.ylim(by[["복사", "모델"]].values.min() - 5, 100)
plt.grid(True, axis="y"); plt.show()
by

In [ ]:
# 성공 사례: 사투리를 정답 표준어와 똑같이 고친 문장
ok = test_s[(test_s.pred.map(norm_key) == test_s.standard.map(norm_key)) & (test_s.dialect != test_s.standard)]
print(f"성공 (정답과 완전일치) {len(ok)}개 중 5개")
for r in ok.head(5).itertuples():
    print(f"  {r.dialect}\n    → {r.pred}   [{changes(r.dialect, r.pred)}]")

# 실패 사례: 문장별 chrF가 가장 낮은 5개 → 발표의 오류 분석 재료
test_s["chrF"] = [sacrebleu.sentence_chrf(p, [r]).score for p, r in zip(test_s.pred, test_s.standard)]
print("\n실패 (chrF 낮은 순) 5개")
for r in test_s.nsmallest(5, "chrF").itertuples():
    print(f"  [{r.chrF:.0f}] 사투리 {r.dialect}\n        정답   {r.standard}\n        모델   {r.pred}")

## 7. 번역 비교 실험 A / B / C
같은 번역 모델(MADLAD 3B)·같은 디코딩 설정으로 세 가지 입력을 중국어로 번역합니다.

| 조건 | 번역기에 넣는 문장 | 의미 |
|---|---|---|
| A | 사투리 원문 그대로 | 기준선 |
| B | 우리 모델이 만든 표준어 | 제안 방법 |
| C | 사람이 쓴 정답 표준어 | 상한선 |

> 시험 세트에는 **중국어 정답이 없어서** 여기서는 "C 번역과 얼마나 비슷한가"(chrF)를 **참고값**으로 봅니다. B가 A보다 C에 가까우면 정규화가 번역을 표준어 쪽으로 끌어온다는 뜻입니다.
> 진짜 점수는 중국어 정답이 있는 평가 세트(`data/eval/eval_set.csv`)가 생기면 아래 마지막 셀에서 계산합니다.

In [ ]:
model.to(dtype=torch.bfloat16 if amp else torch.float32)           # 정규화 모델은 작게 (메모리 확보)
torch.cuda.empty_cache()
mt_tok = AutoTokenizer.from_pretrained(MT_MODEL)
mt = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL).to(device, dtype=amp or torch.float32).eval()  # MADLAD는 fp16에서 넘칠 수 있어 bf16

@torch.inference_mode()
def translate(texts, tgt="zh", bs=2):
    # MADLAD: 입력 앞에 <2zh> 같은 언어 태그를 붙이면 그 언어로 번역
    outs = []
    for i in range(0, len(texts), bs):
        batch = [f"<2{tgt}> {t}" for t in texts[i:i + bs]]
        enc = mt_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        outs += [o.strip() for o in mt_tok.batch_decode(mt.generate(**enc, **MT_DECODE), skip_special_tokens=True)]
        if device == "cuda":
            torch.cuda.empty_cache()   # 8문장씩 묶으면 8GB GPU가 넘쳐서 수십 배 느려짐 → 2문장씩, 묶음마다 메모리 정리
    return outs

print(translate(["오늘 날씨가 좋네요."]))

In [ ]:
abc = test_s.groupby("subset").sample(25, random_state=0, replace=False).reset_index(drop=True)   # 종류별 25문장 = 100문장
t0 = time.time()
abc["zh_A"] = translate(abc.dialect.tolist())     # A: 사투리 그대로
abc["zh_B"] = translate(abc.pred.tolist())        # B: 모델 표준어
abc["zh_C"] = translate(abc.standard.tolist())    # C: 정답 표준어
print(f"번역 {len(abc) * 3}문장 {time.time() - t0:.0f}초")

ref = abc.zh_C.tolist()
print("참고: C 번역과의 chrF (높을수록 C에 가까움)")
print("  A 사투리 그대로:", round(sacrebleu.corpus_chrf(abc.zh_A.tolist(), [ref]).score, 2))
print("  B 모델 표준어  :", round(sacrebleu.corpus_chrf(abc.zh_B.tolist(), [ref]).score, 2))
abc[["dialect", "pred", "zh_A", "zh_B", "zh_C"]].head(10)

In [ ]:
# 문장별로 A와 B 중 어느 쪽이 C에 더 가까운가
abc["A≈C"] = [sacrebleu.sentence_chrf(h, [r]).score for h, r in zip(abc.zh_A, ref)]
abc["B≈C"] = [sacrebleu.sentence_chrf(h, [r]).score for h, r in zip(abc.zh_B, ref)]
win = pd.Series(np.sign(abc["B≈C"] - abc["A≈C"])).map({1: "B가 더 가까움", 0: "같음", -1: "A가 더 가까움"}).value_counts()
print(win)
abc.assign(종류=abc.subset.map(SUBSET_NAME)).groupby("종류")[["A≈C", "B≈C"]].mean() \
   .plot.bar(figsize=(7, 4), rot=0, color=["gray", "red"])
plt.ylabel("C 번역 대비 chrF (참고)"); plt.title("종류별 A vs B"); plt.grid(True, axis="y"); plt.show()

Path("results").mkdir(exist_ok=True)
abc.to_csv("results/notebook_abc.csv", index=False, encoding="utf-8-sig")   # 엑셀로 열어서 보기 (커밋 금지)

In [ ]:
# (평가 세트가 생기면) 중국어 정답으로 진짜 점수 — evaluate.py와 같은 계산
ev_path = Path("data/eval/eval_set.csv")
if ev_path.exists():
    ev = pd.read_csv(ev_path).fillna("")
    ev["pred_standard"] = normalize(ev.dialect.tolist())
    res = {}
    for c, col in [("A", "dialect"), ("B", "pred_standard"), ("C", "standard")]:
        hyp = translate(ev[col].tolist())
        res[c] = {"BLEU_zh": round(sacrebleu.corpus_bleu(hyp, [ev.zh.tolist()], tokenize="zh").score, 2),
                  "chrF": round(sacrebleu.corpus_chrf(hyp, [ev.zh.tolist()]).score, 2)}
    res = pd.DataFrame(res).T
    a, b, c = res.chrF
    if c != a:
        print(f"B가 A→C 격차의 {(b - a) / (c - a) * 100:.1f}%를 메움 (chrF 기준)")
    display(res)
else:
    print("평가 세트(data/eval/eval_set.csv)가 아직 없음 → 생기면 이 셀을 다시 실행")

## 8. Gradio 데모 — 사투리 → 표준어 → 번역을 한 화면에
아래 셀을 실행하면 노트북 안에 화면이 뜹니다 (브라우저로 열려면 `http://127.0.0.1:7860`). 저장소의 `app.py`가 같은 화면을 독립 프로그램으로 띄웁니다.

In [ ]:
import gradio as gr

LANGS = {"중국어 (간체)": "zh", "영어": "en", "일본어": "ja", "베트남어": "vi"}

def demo_fn(text, lang):
    lines = [s.strip() for s in str(text).splitlines() if s.strip()]
    if not lines:
        return "", "", ""
    std = normalize(lines)                                  # ① 우리 모델: 사투리 → 표준어
    tgt = LANGS[lang]
    return "\n".join(std), "\n".join(translate(std, tgt)), "\n".join(translate(lines, tgt))   # ② 표준어 번역, 비교: 직접 번역

with gr.Blocks(title="13조 사투리 번역기") as demo:
    gr.Markdown("### 경상도 사투리 → 표준어 → 외국어")
    with gr.Row():
        with gr.Column():
            inp = gr.Textbox(label="사투리 입력", lines=3, placeholder="예: 니 밥 뭇나?")
            lang = gr.Dropdown(list(LANGS), value="중국어 (간체)", label="번역할 언어")
            btn = gr.Button("번역", variant="primary")
            gr.Examples([["니 밥 뭇나?"], ["어데 가노? 퍼뜩 온나."], ["단디 해라, 알았제?"]], inputs=[inp])
        with gr.Column():
            o1 = gr.Textbox(label="① 표준어 변환 (우리 모델)")
            o2 = gr.Textbox(label="② 표준어 → 번역")
            o3 = gr.Textbox(label="비교: 사투리를 그대로 번역")
    btn.click(demo_fn, [inp, lang], [o1, o2, o3])

demo.launch()          # 끝낼 때: demo.close()

## 9. (도전) 학습률을 바꿔 보면? — 너무 작으면 느리고, 너무 크면 불안정
강의의 학습률 실험과 같은 방법입니다. 사전학습 모델을 **새로** 불러와 2000쌍으로 150스텝만 학습하며 손실을 비교합니다 (학습률마다 2~3분).
> 번역 모델·정규화 모델·데모를 메모리에서 내린 뒤 실행합니다. 이 셀 다음에 다시 번역하려면 `TRAIN = False`로 바꾸고 5장부터 다시 실행하세요.

In [ ]:
try:
    demo.close()
except Exception:
    pass
for name in ("mt", "model"):
    globals().pop(name, None)                               # 큰 모델을 메모리에서 내림
torch.cuda.empty_cache()

small = train_df.sample(n=2000, random_state=7)
curves = {}
for lr_try in [1e-5, 5e-5, 3e-4]:
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr_try)
    hist, step = [], 0
    model.train()
    while step < 150:
        for x, m, y in batches(small.dialect.tolist(), small.standard.tolist(), BATCH):
            with torch.autocast("cuda", dtype=amp, enabled=amp is not None):
                loss = model(input_ids=x, attention_mask=m, labels=y).loss
            loss.backward(); opt.step(); opt.zero_grad()
            hist.append(loss.item()); step += 1
            if step >= 150:
                break
    curves[lr_try] = hist
    print(f"lr = {lr_try:<7}  150스텝 후 손실(최근 20스텝 평균) = {np.mean(hist[-20:]):.4f}")
    del model, opt; torch.cuda.empty_cache()

plt.figure(figsize=(7, 4))
for lr_try, h in curves.items():
    plt.plot(pd.Series(h).rolling(10, min_periods=1).mean(), label=f"lr = {lr_try}")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss (log, 10스텝 이동평균)")
plt.title("학습률 비교: 너무 작음 / 적당 / 너무 큼"); plt.legend(); plt.grid(True); plt.show()

---
### 정리
| 단계 | 강의 (Lecture 11 선형 회귀) | 우리 프로젝트 |
|---|---|---|
| 데이터 | `Salary_Data.csv` 30개 | AI Hub 경상도 방언 → 사투리·표준어 쌍 56만 개 |
| 분할 | `train_test_split` (무작위) | **화자 기준** 훈련·검증·시험 (같은 사람 문장이 섞이지 않게) |
| 모델 | `LinearRegression()` (파라미터 2개) | 사전학습 KoBART (파라미터 약 1.2억 개) |
| 손실 | MSE | 교차 엔트로피 |
| 학습 | 기울기 직접 계산 → `w -= lr * g` | `loss.backward()` → AdamW `optimizer.step()` |
| 예측 | `predict(X_test)` | `generate()` (빔 서치) |
| 평가 | MSE·RMSE·R² | chrF·BLEU·CER + **복사 기준선** 비교 |
| 응용 | — | 표준어 → MADLAD 번역, A/B/C 비교, Gradio 데모 |

같은 내용이 독립 실행용으로도 있습니다: `scripts/train_normalizer.py`(학습), `scripts/predict_normalizer.py`(변환), `scripts/evaluate.py`(A/B/C 점수), `app.py`(데모).